In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

In [5]:
project_dir = Path.cwd().parent
raw_dir = project_dir / "data" / "raw"

csv_files = list(raw_dir.glob("*.csv"))
print("CSV files found:", [file.name for file in csv_files])

if len(csv_files) != 1:
    raise ValueError(
        "Expected one CSV file in data/raw. "
        "Check the folder and select the correct file."
    )

df = pd.read_csv(csv_files[0])

print("Dataset loaded successfully!")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

df.head()

CSV files found: ['final_shop_6modata.csv']
Dataset loaded successfully!
Rows: 190
Columns: 12


,Ad Group,Month,Impressions,Clicks,CTR,Conversions,Conv Rate,Cost,CPC,Revenue,Sale Amount,P&L
0,Shop - 1:1 - Desk - [shop coupon code],July,16038,6504,0.41,1166,0.10,6669,1.03,6402,136770.05,-267.086
1,Shop - 1:1 - Desk - [shop coupon],July,36462,14367,0.39,2188,0.09,13746,0.96,13262,283215.21,-483.951
2,Shop - 1:1 - Desk - [shop discount code],July,3635,1458,0.40,248,0.09,1606,1.10,1723,39165.46,117.136
3,Shop - 1:1 - Desk - [shop promo code],July,26185,10418,0.40,2294,0.12,13278,1.27,13042,284823.48,-235.921
4,Shop - 1:1 - Desk - [shop promo],July,808,282,0.35,61,0.15,391,1.39,337,7717.77,-53.604


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 190 entries, 0 to 189
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Ad Group     190 non-null    object 
 1   Month        190 non-null    object 
 2   Impressions  190 non-null    int64  
 3   Clicks       190 non-null    int64  
 4   CTR          190 non-null    float64
 5   Conversions  190 non-null    int64  
 6   Conv Rate    190 non-null    float64
 7   Cost         190 non-null    int64  
 8   CPC          190 non-null    float64
 9   Revenue      190 non-null    int64  
 10  Sale Amount  190 non-null    float64
 11  P&L          190 non-null    float64
dtypes: float64(5), int64(5), object(2)
memory usage: 17.9+ KB


In [10]:
df.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Ad Group,190,40,Shop - 1:1 - Desk - [shop coupon code],5,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Month,190,5,November,39,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Impressions,190.0,NaN,NaN,NaN,14077.363158,29771.686227,35.0,1065.0,4969.0,13380.0,276568.0
Clicks,190.0,NaN,NaN,NaN,4865.805263,11348.529219,2.0,264.5,930.0,4190.5,99526.0
CTR,190.0,NaN,NaN,NaN,0.272105,0.107894,0.05,0.1925,0.285,0.36,0.47
Conversions,190.0,NaN,NaN,NaN,505.242105,1052.202922,0.0,24.0,70.5,428.25,7563.0
Conv Rate,190.0,NaN,NaN,NaN,0.079737,0.052859,0.0,0.05,0.07,0.1,0.5
Cost,190.0,NaN,NaN,NaN,3344.063158,6524.606753,1.0,188.5,563.5,2967.0,43542.0
CPC,190.0,NaN,NaN,NaN,0.791263,0.403312,0.14,0.48,0.635,1.1075,2.02
Revenue,190.0,NaN,NaN,NaN,2957.684211,5962.413097,0.0,144.5,459.5,2672.25,42440.0


In [11]:

print("Missing values:")
print(df.isna().sum())

print("\nDuplicate rows:")
print(df.duplicated().sum())

print("\nColumn names:")
print(df.columns.tolist())

Missing values:
Ad Group       0
Month          0
Impressions    0
Clicks         0
CTR            0
Conversions    0
Conv Rate      0
Cost           0
CPC            0
Revenue        0
Sale Amount    0
P&L            0
dtype: int64

Duplicate rows:
0

Column names:
['Ad Group', 'Month', 'Impressions', 'Clicks', 'CTR', 'Conversions', 'Conv Rate', 'Cost', 'CPC', 'Revenue', 'Sale Amount', 'P&L']


In [12]:

df_clean = df.copy()

df_clean.columns = (
    df_clean.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

df_clean = df_clean.rename(columns={
    "conv_rate": "conversion_rate",
    "p&l": "profit_loss"
})

print(df_clean.columns.tolist())


['ad_group', 'month', 'impressions', 'clicks', 'ctr', 'conversions', 'conversion_rate', 'cost', 'cpc', 'revenue', 'sale_amount', 'profit_loss']


In [13]:

print(df_clean["month"].unique())

print(
    df_clean.groupby("month")
    .size()
    .rename("row_count")
)


['July' 'August' 'September' 'October' 'November']
month
August       38
July         38
November     39
October      37
September    38
Name: row_count, dtype: int64


In [14]:

duplicate_keys = df_clean[
    df_clean.duplicated(
        subset=["ad_group", "month"],
        keep=False
    )
]

print("Repeated ad group-month combinations:", len(duplicate_keys))

display(
    duplicate_keys.sort_values(["ad_group", "month"])
)


Repeated ad group-month combinations: 0


,ad_group,month,impressions,clicks,ctr,conversions,conversion_rate,cost,cpc,revenue,sale_amount,profit_loss


In [15]:

checks = {
    "Negative impressions":
        (df_clean["impressions"] < 0).sum(),

    "Negative clicks":
        (df_clean["clicks"] < 0).sum(),

    "Negative conversions":
        (df_clean["conversions"] < 0).sum(),

    "Negative cost":
        (df_clean["cost"] < 0).sum(),

    "Clicks greater than impressions":
        (df_clean["clicks"] > df_clean["impressions"]).sum(),

    "CTR outside 0-1":
        ((df_clean["ctr"] < 0) |
         (df_clean["ctr"] > 1)).sum(),

    "Conversion rate outside 0-1":
        ((df_clean["conversion_rate"] < 0) |
         (df_clean["conversion_rate"] > 1)).sum()
}

pd.Series(checks, name="Issue count")


Negative impressions               0
Negative clicks                    0
Negative conversions               0
Negative cost                      0
Clicks greater than impressions    0
CTR outside 0-1                    0
Conversion rate outside 0-1        0
Name: Issue count, dtype: int64

In [16]:

checks = {
    "Negative impressions":
        (df_clean["impressions"] < 0).sum(),

    "Negative clicks":
        (df_clean["clicks"] < 0).sum(),

    "Negative conversions":
        (df_clean["conversions"] < 0).sum(),

    "Negative cost":
        (df_clean["cost"] < 0).sum(),

    "Clicks greater than impressions":
        (df_clean["clicks"] > df_clean["impressions"]).sum(),

    "CTR outside 0-1":
        ((df_clean["ctr"] < 0) |
         (df_clean["ctr"] > 1)).sum(),

    "Conversion rate outside 0-1":
        ((df_clean["conversion_rate"] < 0) |
         (df_clean["conversion_rate"] > 1)).sum()
}

pd.Series(checks, name="Issue count")


Negative impressions               0
Negative clicks                    0
Negative conversions               0
Negative cost                      0
Clicks greater than impressions    0
CTR outside 0-1                    0
Conversion rate outside 0-1        0
Name: Issue count, dtype: int64

In [17]:

df_clean["calculated_ctr"] = np.where(
    df_clean["impressions"] > 0,
    df_clean["clicks"] / df_clean["impressions"],
    np.nan
)

df_clean["calculated_conversion_rate"] = np.where(
    df_clean["clicks"] > 0,
    df_clean["conversions"] / df_clean["clicks"],
    np.nan
)

df_clean["calculated_cpc"] = np.where(
    df_clean["clicks"] > 0,
    df_clean["cost"] / df_clean["clicks"],
    np.nan
)

display(
    df_clean[
        [
            "ctr", "calculated_ctr",
            "conversion_rate", "calculated_conversion_rate",
            "cpc", "calculated_cpc"
        ]
    ].head(10)
)


,ctr,calculated_ctr,conversion_rate,calculated_conversion_rate,cpc,calculated_cpc
0,0.41,0.405537,0.10,0.179274,1.03,1.025369
1,0.39,0.394027,0.09,0.152293,0.96,0.956776
2,0.40,0.401100,0.09,0.170096,1.10,1.101509
3,0.40,0.397861,0.12,0.220196,1.27,1.274525
4,0.35,0.349010,0.15,0.216312,1.39,1.386525
5,0.47,0.467801,0.05,0.076531,0.60,0.604753
6,0.39,0.388316,0.04,0.045051,0.33,0.327340
7,0.43,0.430452,0.05,0.081018,0.62,0.615690
8,0.47,0.472714,0.07,0.108403,0.62,0.624829
9,0.38,0.376533,0.07,0.097734,0.69,0.686969


In [18]:

for original, calculated in [
    ("ctr", "calculated_ctr"),
    ("conversion_rate", "calculated_conversion_rate"),
    ("cpc", "calculated_cpc")
]:
    difference = (
        df_clean[original] - df_clean[calculated]
    ).abs()

    print(
        f"{original}: "
        f"max difference = {difference.max():.6f}"
    )


ctr: max difference = 0.005000
conversion_rate: max difference = 0.250000
cpc: max difference = 0.286667


In [19]:

df_clean["calculated_profit_loss"] = (
    df_clean["revenue"] - df_clean["cost"]
)

df_clean["profit_difference"] = (
    df_clean["profit_loss"]
    - df_clean["calculated_profit_loss"]
)

print(
    df_clean["profit_difference"]
    .abs()
    .describe()
)

display(
    df_clean.loc[
        df_clean["profit_difference"].abs() > 1,
        [
            "ad_group", "month", "cost",
            "revenue", "profit_loss",
            "calculated_profit_loss"
        ]
    ].head(10)
)


count    190.000000
mean       0.228011
std        0.149414
min        0.000000
25%        0.100250
50%        0.223000
75%        0.350500
max        0.499000
Name: profit_difference, dtype: float64


,ad_group,month,cost,revenue,profit_loss,calculated_profit_loss


In [20]:

# Compare supplied metrics against calculated metrics
kpi_checks = {
    "CTR": (
        df_clean["ctr"] -
        df_clean["calculated_ctr"]
    ).abs() > 0.0051,

    "Conversion Rate": (
        df_clean["conversion_rate"] -
        df_clean["calculated_conversion_rate"]
    ).abs() > 0.0051,

    "CPC": (
        df_clean["cpc"] -
        df_clean["calculated_cpc"]
    ).abs() > 0.0051
}

for metric, mismatch in kpi_checks.items():
    print(f"{metric}: {mismatch.sum()} mismatched records")

# Examine the largest conversion rate discrepancies
df_clean["conversion_rate_difference"] = (
    df_clean["conversion_rate"] -
    df_clean["calculated_conversion_rate"]
).abs()

display(
    df_clean.sort_values(
        "conversion_rate_difference",
        ascending=False
    )[
        [
            "ad_group",
            "month",
            "clicks",
            "conversions",
            "conversion_rate",
            "calculated_conversion_rate"
        ]
    ].head(10)
)


CTR: 0 mismatched records
Conversion Rate: 160 mismatched records
CPC: 28 mismatched records


,ad_group,month,clicks,conversions,conversion_rate,calculated_conversion_rate
112,Shop - Phrase - Desk - Free Shipping,September,4,1,0.50,0.250000
60,Shop - Exact - Desk - Offer,August,163,39,0.12,0.239264
116,Shop - 1:1 - Desk - [shop promo code],October,6484,1499,0.12,0.231184
59,Shop - 1:1 - Desk - [shop promo],August,240,52,0.11,0.216667
14,Shop - Exact - Desk - Offer,July,115,26,0.12,0.226087
44,Shop - Exact - Desk - Promo Code,August,2328,498,0.11,0.213918
127,Shop - 1:1 - Desk - [shop discount code],October,855,173,0.10,0.202339
39,Shop - 1:1 - Desk - [shop promo code],August,7673,1622,0.11,0.211391
121,Shop - Exact - Desk - Promo Code,October,2064,456,0.12,0.220930
73,Shop - Phrase - Mob - Competitor,August,13,3,0.13,0.230769


In [21]:

df_final = df_clean[
    [
        "ad_group",
        "month",
        "impressions",
        "clicks",
        "conversions",
        "cost",
        "revenue",
        "sale_amount",
        "ctr",
        "conversion_rate",
        "cpc",
        "profit_loss"
    ]
].copy()

# Rename supplied KPI fields to clarify their origin
df_final = df_final.rename(columns={
    "ctr": "source_ctr",
    "conversion_rate": "source_conversion_rate",
    "cpc": "source_cpc",
    "profit_loss": "source_profit_loss"
})

# Calculate consistent reporting metrics
df_final["ctr"] = (
    df_final["clicks"] /
    df_final["impressions"].replace(0, np.nan)
)

df_final["conversion_rate"] = (
    df_final["conversions"] /
    df_final["clicks"].replace(0, np.nan)
)

df_final["cpc"] = (
    df_final["cost"] /
    df_final["clicks"].replace(0, np.nan)
)

df_final["cpa"] = (
    df_final["cost"] /
    df_final["conversions"].replace(0, np.nan)
)

df_final["roas"] = (
    df_final["revenue"] /
    df_final["cost"].replace(0, np.nan)
)

df_final["profit_loss"] = (
    df_final["revenue"] - df_final["cost"]
)

display(df_final.head())


,ad_group,month,impressions,clicks,conversions,cost,revenue,sale_amount,source_ctr,source_conversion_rate,source_cpc,source_profit_loss,ctr,conversion_rate,cpc,cpa,roas,profit_loss
0,Shop - 1:1 - Desk - [shop coupon code],July,16038,6504,1166,6669,6402,136770.05,0.41,0.10,1.03,-267.086,0.405537,0.179274,1.025369,5.719554,0.959964,-267
1,Shop - 1:1 - Desk - [shop coupon],July,36462,14367,2188,13746,13262,283215.21,0.39,0.09,0.96,-483.951,0.394027,0.152293,0.956776,6.282450,0.964790,-484
2,Shop - 1:1 - Desk - [shop discount code],July,3635,1458,248,1606,1723,39165.46,0.40,0.09,1.10,117.136,0.401100,0.170096,1.101509,6.475806,1.072852,117
3,Shop - 1:1 - Desk - [shop promo code],July,26185,10418,2294,13278,13042,284823.48,0.40,0.12,1.27,-235.921,0.397861,0.220196,1.274525,5.788143,0.982226,-236
4,Shop - 1:1 - Desk - [shop promo],July,808,282,61,391,337,7717.77,0.35,0.15,1.39,-53.604,0.349010,0.216312,1.386525,6.409836,0.861893,-54


In [22]:

reporting_year = 2021

df_final["reporting_month"] = pd.to_datetime(
    df_final["month"] + " " + str(reporting_year),
    format="%B %Y"
)

df_final = df_final.sort_values(
    ["reporting_month", "ad_group"]
).reset_index(drop=True)

display(
    df_final[
        ["ad_group", "month", "reporting_month"]
    ].head()
)


,ad_group,month,reporting_month
0,Shop - 1:1 - Desk - [shop coupon code],July,2021-07-01
1,Shop - 1:1 - Desk - [shop coupon],July,2021-07-01
2,Shop - 1:1 - Desk - [shop discount code],July,2021-07-01
3,Shop - 1:1 - Desk - [shop promo code],July,2021-07-01
4,Shop - 1:1 - Desk - [shop promo],July,2021-07-01


In [23]:

processed_dir = project_dir / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

output_path = processed_dir / "advertising_campaigns_clean.csv"

df_final.to_csv(output_path, index=False)

print("Saved cleaned dataset to:")
print(output_path)

print("\nFinal shape:", df_final.shape)
print("Missing values:")
print(df_final.isna().sum())


Saved cleaned dataset to:
/Users/nikhitakarthikesan/Documents/advertising-campaign-analytics/data/processed/advertising_campaigns_clean.csv

Final shape: (190, 19)
Missing values:
ad_group                  0
month                     0
impressions               0
clicks                    0
conversions               0
cost                      0
revenue                   0
sale_amount               0
source_ctr                0
source_conversion_rate    0
source_cpc                0
source_profit_loss        0
ctr                       0
conversion_rate           0
cpc                       0
cpa                       7
roas                      0
profit_loss               0
reporting_month           0
dtype: int64


In [24]:

df_check = pd.read_csv(output_path)

assert len(df_check) == 190
assert df_check.duplicated(
    subset=["ad_group", "reporting_month"]
).sum() == 0

assert (df_check["impressions"] >= 0).all()
assert (df_check["clicks"] >= 0).all()
assert (df_check["cost"] >= 0).all()

print("Export validation passed!")
print("Rows:", len(df_check))
print("Columns:", len(df_check.columns))


Export validation passed!
Rows: 190
Columns: 19
